In [ ]:

# ============================================================
# ASAMA 1 - BASLANGIC
# ============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

GERCEK_LABEL = "Ger\u00e7ek"
TAHIL_LABEL = "Tah\u0131l"


def find_data_dir():
    """Find the repo data directory regardless of the notebook working directory."""
    current = Path.cwd().resolve()
    for base in [current, *current.parents]:
        candidate = base / "veri"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Repo veri klasoru bulunamadi.")


def pick_data_file(pattern):
    matches = sorted(DATA_DIR.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Veri dosyasi bulunamadi: {pattern}")
    return matches[0]


DATA_DIR = find_data_dir()
REPO_ROOT = DATA_DIR.parent
MODEL_DIR = REPO_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

PATH_MEYVE = DATA_DIR / "tuketim_meyve.xlsx"
PATH_SEBZE = DATA_DIR / "tuketim_sebze.xlsx"
PATH_TAHIL = DATA_DIR / "tuketim_tahil.xlsx"
PATH_NUFUS = pick_data_file("Nufus*.xlsx")
PATH_ENDEKS = DATA_DIR / "Endeks_Verileri_2014_2024.xlsx"
PATH_GELIR = pick_data_file("*Hanehalk*.xlsx")

print("Calisma klasoru:", os.getcwd())
print("Veri klasoru:", DATA_DIR)
print("Model cikti klasoru:", MODEL_DIR)

print("\nDosya kontrolu:")
for p in [PATH_MEYVE, PATH_SEBZE, PATH_TAHIL, PATH_NUFUS, PATH_ENDEKS, PATH_GELIR]:
    print(p.name, "->", "VAR" if p.exists() else "YOK")


In [ ]:

# ============================================================
# ASAMA 2 - VERI OKUMA
# ============================================================

meyve = pd.read_excel(PATH_MEYVE)
sebze = pd.read_excel(PATH_SEBZE)
tahil = pd.read_excel(PATH_TAHIL)

nufus = pd.read_excel(PATH_NUFUS, header=1)
endeks = pd.read_excel(PATH_ENDEKS)
gelir = pd.read_excel(PATH_GELIR)

print("MEYVE:", meyve.shape)
print("SEBZE:", sebze.shape)
print("TAHIL:", tahil.shape)
print("NUFUS:", nufus.shape)
print("ENDEKS:", endeks.shape)
print("GELIR:", gelir.shape)


In [ ]:

# ============================================================
# ASAMA 3 - TUKETIM VERISI
# ============================================================

meyve["Ana_Kategori"] = "Meyve"
sebze["Ana_Kategori"] = "Sebze"
tahil["Ana_Kategori"] = "Tahil"

tuketim = pd.concat([meyve, sebze, tahil], ignore_index=True)

tuketim = tuketim.rename(columns={
    "Urun Adi": "Urun_Adi",
    "Deger": "Tuketim"
})

tuketim = tuketim[["Yil", "Urun_Adi", "Tuketim", "Ana_Kategori"]]

print("TUKETIM SHAPE:", tuketim.shape)

print("\nYillara gore satir sayisi:")
print(tuketim.groupby("Yil").size())

tuketim.head()


In [ ]:

# ============================================================
# ASAMA 4 - NUFUS
# ============================================================

# In the source file, the first column is year and the third column is total population.
nufus = nufus.iloc[:, [0, 2]].copy()
nufus.columns = ["Yil", "Nufus"]

nufus["Yil"] = pd.to_numeric(nufus["Yil"], errors="coerce")
nufus["Nufus"] = pd.to_numeric(nufus["Nufus"], errors="coerce")
nufus = nufus.dropna().copy()
nufus["Yil"] = nufus["Yil"].astype(int)

print(nufus)


In [ ]:

# ============================================================
# ASAMA 5 - ENDEKS
# ============================================================

# Annual index is calculated as the average of monthly index values.
endeks = pd.DataFrame({
    "Yil": pd.to_numeric(endeks.iloc[:, 0], errors="coerce"),
    "Endeks_Ortalama": endeks.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").mean(axis=1),
})

endeks = endeks.dropna().copy()
endeks["Yil"] = endeks["Yil"].astype(int)

print(endeks)


In [ ]:

# ============================================================
# ASAMA 6 - GELIR
# ============================================================

gelir = gelir.iloc[3:, [2, 3]].copy()
gelir.columns = ["Yil", "Hanehalki_Gelir"]

gelir["Yil"] = pd.to_numeric(gelir["Yil"], errors="coerce")
gelir["Hanehalki_Gelir"] = pd.to_numeric(gelir["Hanehalki_Gelir"], errors="coerce")
gelir = gelir.dropna().copy()
gelir["Yil"] = gelir["Yil"].astype(int)

# Some files contain multiple income types for the same year; use yearly average.
gelir = gelir.groupby("Yil", as_index=False)["Hanehalki_Gelir"].mean()

print(gelir)


In [ ]:

# ============================================================
# ASAMA 7 - DATASET
# ============================================================

data = tuketim.merge(nufus, on="Yil", how="left")
data = data.merge(endeks, on="Yil", how="left")
data = data.merge(gelir, on="Yil", how="left")

print("DATA SHAPE:", data.shape)

print("\nYillara gore satir sayisi:")
print(data.groupby("Yil").size())

data.head()


In [ ]:

# ============================================================
# ASAMA 8 - FEATURE ENGINEERING
# ============================================================

df = data.copy()

df = df.sort_values(["Urun_Adi", "Yil"]).reset_index(drop=True)

grp = df.groupby("Urun_Adi", group_keys=False)

# Previous-year consumption.
df["Lag1"] = grp["Tuketim"].shift(1)

# Product-level trailing 3-year mean. transform prevents cross-product rolling leakage.
df["RollingMean3"] = grp["Tuketim"].transform(lambda s: s.shift(1).rolling(3).mean())

# Log transforms.
df["Lag1_log"] = np.log1p(df["Lag1"])
df["RollingMean3_log"] = np.log1p(df["RollingMean3"])

# Time trend.
df["Trend"] = df["Yil"] - df["Yil"].min()

print("Feature dataset shape:", df.shape)

print("\nEksik degerler:")
print(df[["Lag1", "RollingMean3", "Nufus", "Endeks_Ortalama", "Hanehalki_Gelir"]].isna().sum())

print("\nOrnek veri:")
display(df.head(10))


In [ ]:

# ============================================================
# ASAMA 9 - MODEL DATASET
# ============================================================

model_df = df.dropna().copy()

print("Model dataset shape:", model_df.shape)

print("\nYil araligi:")
print(model_df["Yil"].min(), "-", model_df["Yil"].max())

print("\nKategori bazinda urun sayisi:")
print(model_df.groupby("Ana_Kategori")["Urun_Adi"].nunique())

print("\nOrnek veri:")
display(model_df.head(10))


In [ ]:

# ============================================================
# ASAMA 10 - TRAIN TEST
# ============================================================

train_df = model_df[model_df["Yil"] < 2023]
test_df = model_df[model_df["Yil"] == 2023]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain yillari:", sorted(train_df["Yil"].unique()))
print("Test yili:", sorted(test_df["Yil"].unique()))


In [ ]:

# ============================================================
# ASAMA 11 - MODEL
# ============================================================

FEATURES_NUM = [
    "Lag1_log",
    "RollingMean3_log",
    "Trend",
    "Nufus",
    "Endeks_Ortalama",
    "Hanehalki_Gelir",
]

FEATURES_CAT = [
    "Urun_Adi",
]

TARGET = "Tuketim"


def _wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100)


def _smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    out = np.zeros_like(y_true, dtype=float)
    out[mask] = 2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]
    return float(np.mean(out) * 100)


def build_model(alpha):
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), FEATURES_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
    ])
    return Pipeline([
        ("prep", preprocess),
        ("model", Ridge(alpha=alpha)),
    ])


# Hyperparameter is selected on earlier validation years, not on the final 2023 holdout.
alpha_candidates = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
validation_years = [2020, 2021, 2022]
alpha_rows = []

for alpha in alpha_candidates:
    fold_scores = []
    for validation_year in validation_years:
        fold_train = model_df[model_df["Yil"] < validation_year]
        fold_valid = model_df[model_df["Yil"] == validation_year]
        if len(fold_train) == 0 or len(fold_valid) == 0:
            continue

        candidate_model = build_model(alpha)
        candidate_model.fit(fold_train[FEATURES_NUM + FEATURES_CAT], np.log1p(fold_train[TARGET]))
        fold_pred = np.expm1(candidate_model.predict(fold_valid[FEATURES_NUM + FEATURES_CAT]))
        fold_scores.append({
            "validation_year": validation_year,
            "WAPE_%": _wape(fold_valid[TARGET], fold_pred),
            "SMAPE_%": _smape(fold_valid[TARGET], fold_pred),
            "R2": r2_score(fold_valid[TARGET], fold_pred),
        })

    if fold_scores:
        alpha_rows.append({
            "alpha": alpha,
            "mean_WAPE_%": float(np.mean([row["WAPE_%"] for row in fold_scores])),
            "mean_SMAPE_%": float(np.mean([row["SMAPE_%"] for row in fold_scores])),
            "mean_R2": float(np.mean([row["R2"] for row in fold_scores])),
        })

alpha_summary = pd.DataFrame(alpha_rows).sort_values(["mean_WAPE_%", "mean_SMAPE_%"]).reset_index(drop=True)
RIDGE_ALPHA = float(alpha_summary.iloc[0]["alpha"]) if len(alpha_summary) else 10.0

print("Secilen Ridge alpha:", RIDGE_ALPHA)
display(alpha_summary)

model = build_model(RIDGE_ALPHA)


In [ ]:

# ============================================================
# ASAMA 12 - MODEL EGIT
# ============================================================

X_train = train_df[FEATURES_NUM + FEATURES_CAT]
y_train = np.log1p(train_df[TARGET])

X_test = test_df[FEATURES_NUM + FEATURES_CAT]
y_test = test_df[TARGET]

model.fit(X_train, y_train)

pred_log = model.predict(X_test)
pred = np.expm1(pred_log)


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    out = np.zeros_like(y_true, dtype=float)
    out[mask] = 2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]
    return float(np.mean(out) * 100)


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100)


print("R2:", round(r2_score(y_test, pred), 4))
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print("SMAPE:", round(smape(y_test, pred), 2))
print("WAPE:", round(wape(y_test, pred), 2))


In [ ]:

# ============================================================
# ASAMA 13 - MAKRO VERILER
# ============================================================

macro = nufus.merge(endeks, on="Yil", how="outer").merge(gelir, on="Yil", how="outer")
macro = macro.sort_values("Yil").reset_index(drop=True)

origin_year = 2023
future_years = [2024, 2025, 2026, 2027]

# Macro projections use only data available up to 2023 to avoid look-ahead leakage.
historical_macro = macro[macro["Yil"] <= origin_year].dropna(
    subset=["Nufus", "Endeks_Ortalama", "Hanehalki_Gelir"]
).copy()

macro_future = historical_macro.copy()

for col in ["Nufus", "Endeks_Ortalama", "Hanehalki_Gelir"]:
    lr = LinearRegression()
    lr.fit(historical_macro[["Yil"]], historical_macro[col])

    future_frame = pd.DataFrame({"Yil": future_years})
    future_values = lr.predict(future_frame[["Yil"]])

    for year, value in zip(future_years, future_values):
        if year not in macro_future["Yil"].values:
            macro_future = pd.concat([macro_future, pd.DataFrame({"Yil": [year]})], ignore_index=True)
        macro_future.loc[macro_future["Yil"] == year, col] = value

macro_future = macro_future.sort_values("Yil").reset_index(drop=True)

macro_future.tail(10)


In [ ]:

# ============================================================
# ASAMA 14 - RECURSIVE FORECAST
# ============================================================

future_years = [2024, 2025, 2026, 2027]

predictions = []

for urun, g in df.groupby("Urun_Adi"):

    g = g.sort_values("Yil").copy()

    last_lag = float(g.iloc[-1]["Tuketim"])
    rolling = float(g["Tuketim"].tail(3).mean())

    for year in future_years:

        macro_row = macro_future.loc[macro_future["Yil"] == year].iloc[0]

        row = pd.DataFrame({
            "Lag1_log": [np.log1p(last_lag)],
            "RollingMean3_log": [np.log1p(rolling)],
            "Trend": [year - df["Yil"].min()],
            "Nufus": [macro_row["Nufus"]],
            "Endeks_Ortalama": [macro_row["Endeks_Ortalama"]],
            "Hanehalki_Gelir": [macro_row["Hanehalki_Gelir"]],
            "Urun_Adi": [urun],
        })

        pred_log = model.predict(row)[0]
        pred = np.expm1(pred_log)

        # Stabilizer.
        pred = 0.7 * pred + 0.3 * rolling

        # Conservative band to reduce extreme recursive jumps.
        pred = np.clip(pred, rolling * 0.5, rolling * 1.8)

        predictions.append({
            "Urun_Adi": urun,
            "Yil": year,
            "Tahmin_Tuketim": pred,
        })

        last_lag = pred
        rolling = (rolling * 2 + pred) / 3


pred_df = pd.DataFrame(predictions)

print(pred_df.shape)
pred_df.head(10)


In [ ]:

# ============================================================
# ASAMA 15 - COMBINE
# ============================================================

real = df[["Yil", "Urun_Adi", "Tuketim"]].copy()
real["Tip"] = GERCEK_LABEL

pred_df["Tip"] = "Tahmin"
pred_df = pred_df.rename(columns={"Tahmin_Tuketim": "Tuketim"})

combined = pd.concat([real, pred_df])

combined.head()


In [ ]:
# ============================================================
# AŞAMA 16 — GRAFİKLER
# ============================================================

import os
import matplotlib.pyplot as plt

os.makedirs("grafiklerv3", exist_ok=True)

for urun, g in combined.groupby("Urun_Adi"):

    g = g.sort_values("Yil")

    real = g[g["Tip"]=="Gerçek"]
    pred = g[g["Tip"]=="Tahmin"]

    plt.figure(figsize=(8,4))

    plt.plot(real["Yil"], real["Tuketim"], marker="o", label="Gerçek")
    plt.plot(pred["Yil"], pred["Tuketim"], marker="o", label="Tahmin")

    plt.title(urun)
    plt.xlabel("Yıl")
    plt.ylabel("Tüketim")

    plt.legend()

    plt.tight_layout()

    plt.savefig(f"grafiklerv3/{urun}.png")

    plt.close()

print("Grafikler hazır.")

In [ ]:
# ============================================================
# TÜM GRAFİKLER TEK PDF
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = "tuketim_grafikv3.pdf"

with PdfPages(pdf_path) as pdf:

    for urun, g in combined.groupby("Urun_Adi"):

        g = g.sort_values("Yil")

        real = g[g["Tip"]=="Gerçek"]
        pred = g[g["Tip"]=="Tahmin"]

        plt.figure(figsize=(10,5))

        plt.plot(real["Yil"], real["Tuketim"], marker="o", label="Gerçek")
        plt.plot(pred["Yil"], pred["Tuketim"], marker="o", linestyle="--", label="Tahmin")

        plt.title(f"{urun} Tüketim Tahmini (2014-2027)")
        plt.xlabel("Yıl")
        plt.ylabel("Tüketim (Ton)")

        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        pdf.savefig()
        plt.close()

print("PDF oluşturuldu:", pdf_path)

In [ ]:

# ============================================================
# EXCEL CIKTISI
# ============================================================

# Standardize forecast column name.
pred_export = pred_df.copy()
pred_export = pred_export.rename(columns={"Tahmin_Tuketim": "Tuketim"})
pred_export["Tip"] = "Tahmin"

output_file = MODEL_DIR / "tuketim_tahminleri_2024_2027v3.xlsx"
combined_output_file = MODEL_DIR / "tuketim_ve_tahmin_birlesik.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    pred_export.sort_values(["Urun_Adi", "Yil"]).to_excel(
        writer,
        sheet_name="Tahminler",
        index=False,
    )

    df.sort_values(["Urun_Adi", "Yil"]).to_excel(
        writer,
        sheet_name="Gercek_Veriler",
        index=False,
    )

    combined.sort_values(["Urun_Adi", "Yil"]).to_excel(
        writer,
        sheet_name="Tum_Veriler",
        index=False,
    )

actual_sheets = []
for source_name, label in [
    ("tuketim_meyve.xlsx", "Meyve"),
    ("tuketim_sebze.xlsx", "Sebze"),
    ("tuketim_tahil.xlsx", TAHIL_LABEL),
]:
    actual = pd.read_excel(DATA_DIR / source_name).rename(columns={"Urun Adi": "Urun_Adi", "Deger": "Tuketim"})
    actual = actual[["Yil", "Urun_Adi", "Tuketim"]].copy()
    actual["Tip"] = GERCEK_LABEL
    actual["Kaynak"] = label
    actual_sheets.append((label, actual))

all_actual = pd.concat([sheet for _, sheet in actual_sheets], ignore_index=True)
gercek_birlesik = all_actual[["Yil", "Urun_Adi", "Tip", "Tuketim"]].copy()
gercek_tahmin_birlesik = pd.concat(
    [gercek_birlesik, pred_export[["Yil", "Urun_Adi", "Tip", "Tuketim"]]],
    ignore_index=True,
)

with pd.ExcelWriter(combined_output_file, engine="openpyxl") as writer:
    for sheet_label, actual in actual_sheets:
        sheet_name = {"Meyve": "Meyve_Gercek", "Sebze": "Sebze_Gercek", TAHIL_LABEL: "Tahil_Gercek"}[sheet_label]
        actual.sort_values(["Urun_Adi", "Yil"]).to_excel(writer, sheet_name=sheet_name, index=False)

    gercek_birlesik.sort_values(["Urun_Adi", "Yil"]).to_excel(writer, sheet_name="Gercek_Birlesik", index=False)
    pred_export[["Yil", "Urun_Adi", "Tuketim", "Tip"]].sort_values(["Urun_Adi", "Yil"]).to_excel(writer, sheet_name="Tahminler", index=False)
    combined.sort_values(["Urun_Adi", "Yil"]).to_excel(writer, sheet_name="Tahmin_Dosyasi_TumVeri", index=False)
    gercek_tahmin_birlesik.sort_values(["Urun_Adi", "Yil"]).to_excel(writer, sheet_name="Gercek_ve_Tahmin_Birlesik", index=False)

# The running application currently imports these files from veri/. Keep those synced without changing the primary model output folder.
for source_file in [output_file, combined_output_file]:
    target_file = DATA_DIR / source_file.name
    target_file.write_bytes(source_file.read_bytes())

print("Excel dosyalari models klasorune olusturuldu:")
print("-", output_file)
print("-", combined_output_file)
print("Uygulama veri kopyalari da senkron tutuldu:")
print("-", DATA_DIR / output_file.name)
print("-", DATA_DIR / combined_output_file.name)


In [ ]:
# ============================================================
# LEARNING CURVE
# ============================================================

from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

X_all = pd.concat([X_train, X_test])
y_all = np.log1p(pd.concat([train_df["Tuketim"], test_df["Tuketim"]]))

train_sizes, train_scores, val_scores = learning_curve(
    model,
    X_all,
    y_all,
    cv=5,
    scoring="r2",
    train_sizes=np.linspace(0.1,1.0,10),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(8,5))

plt.plot(train_sizes, train_mean, marker="o", label="Training Score")
plt.plot(train_sizes, val_mean, marker="o", label="Validation Score")

plt.title("Learning Curve")
plt.xlabel("Training Size")
plt.ylabel("R² Score")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# ACTUAL VS PREDICTED SCATTER
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Test tahminini yeniden üret
X_test_eval = test_df[FEATURES_NUM + FEATURES_CAT]
y_test_eval = test_df["Tuketim"].values

pred_log_eval = model.predict(X_test_eval)
pred_eval = np.expm1(pred_log_eval)

print("y_test boyutu :", len(y_test_eval))
print("pred boyutu   :", len(pred_eval))

plt.figure(figsize=(6,6))

plt.scatter(y_test_eval, pred_eval, alpha=0.7)

max_val = max(np.max(y_test_eval), np.max(pred_eval))
plt.plot([0, max_val], [0, max_val], color="red")

plt.xlabel("Gerçek Tüketim")
plt.ylabel("Tahmin Edilen Tüketim")
plt.title("Actual vs Predicted")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# RESIDUAL PLOT
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Test tahminini yeniden üret
X_test_eval = test_df[FEATURES_NUM + FEATURES_CAT]
y_test_eval = test_df["Tuketim"].values

pred_log_eval = model.predict(X_test_eval)
pred_eval = np.expm1(pred_log_eval)

residuals = y_test_eval - pred_eval

print("Residual boyutu:", len(residuals))

plt.figure(figsize=(7,5))

plt.scatter(pred_eval, residuals, alpha=0.7)
plt.axhline(0, color="red")

plt.xlabel("Tahmin Edilen Değer")
plt.ylabel("Residual (Hata)")
plt.title("Residual Plot")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# SHAPE + SCATTER + RESIDUAL
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Test verisi
X_test_eval = test_df[FEATURES_NUM + FEATURES_CAT]
y_test_eval = test_df["Tuketim"].values

# Tahmin
pred_log_eval = model.predict(X_test_eval)
pred_eval = np.expm1(pred_log_eval)

# ============================================================
# SHAPE KONTROLÜ
# ============================================================

print("X_test shape :", X_test_eval.shape)
print("y_test shape :", y_test_eval.shape)
print("pred shape   :", pred_eval.shape)

# ============================================================
# SCATTER PLOT
# ============================================================

plt.figure(figsize=(6,6))

plt.scatter(y_test_eval, pred_eval, alpha=0.7)

max_val = max(np.max(y_test_eval), np.max(pred_eval))

plt.plot([0,max_val],[0,max_val], color="red")

plt.xlabel("Gerçek Tüketim")
plt.ylabel("Tahmin Edilen Tüketim")
plt.title("Actual vs Predicted")

plt.grid(alpha=0.3)

plt.show()

# ============================================================
# RESIDUAL PLOT
# ============================================================

residuals = y_test_eval - pred_eval

print("Residual shape:", residuals.shape)

plt.figure(figsize=(7,5))

plt.scatter(pred_eval, residuals, alpha=0.7)

plt.axhline(0, color="red")

plt.xlabel("Tahmin Edilen Değer")
plt.ylabel("Residual (Hata)")

plt.title("Residual Plot")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# SHAP ANALİZİ (DÜZELTİLMİŞ)
# ============================================================

import shap
import pandas as pd
import numpy as np

# örnek veri
X_sample = X_train.sample(min(200, len(X_train)), random_state=42)

# preprocessing sonrası matris
X_transformed = model.named_steps["prep"].transform(X_sample)

# sparse ise dense'e çevir
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

# feature isimleri
num_features = FEATURES_NUM
cat_encoder = model.named_steps["prep"].named_transformers_["cat"]
cat_features = cat_encoder.get_feature_names_out(FEATURES_CAT)
feature_names = list(num_features) + list(cat_features)

print("X_transformed shape:", X_transformed.shape)
print("Feature name sayısı:", len(feature_names))

# dataframe
X_transformed_df = pd.DataFrame(X_transformed, columns=feature_names)

# ridge modeli
ridge_model = model.named_steps["model"]

# shap
explainer = shap.Explainer(ridge_model, X_transformed_df)
shap_values = explainer(X_transformed_df)

# önem grafiği
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)